# BioTransport CFD Lab

Learning goals: connect random walks, diffusion, advection, reaction, radial geometry, and biosensor response to biological transport examples. The Python solvers are fast teaching models with prescribed flow fields. OpenFOAM is optional for higher-fidelity CFD.

In [ ]:
# Setup cell
import tempfile
import sys
from pathlib import Path

import matplotlib.pyplot as plt

try:
    from IPython.display import Image, display
except ImportError:
    Image = None

    def display(value):
        print(value)

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

try:
    import biotransport_lab  # noqa: F401
except ImportError:
    # Running standalone (e.g. Google Colab) without the cloned repo: install the package from GitHub.
    import subprocess

    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "bio-transport-cfd-lab @ git+https://github.com/HyeonjeYang/bio-transport-cfd-lab.git"],
        check=True,
    )

# Figures are written to a throwaway temp dir just so they can be displayed inline; nothing is kept.
OUT = Path(tempfile.mkdtemp(prefix="biotransport_notebook_"))

from biotransport_lab.api import simulate_cartesian_preset, simulate_radial_preset  # noqa: E402
from biotransport_lab.dimensionless import summarize_dimensionless_numbers  # noqa: E402
from biotransport_lab.random_walk import simulate_random_walk  # noqa: E402
from biotransport_lab.visualization import (  # noqa: E402
    save_cartesian_outputs,
    save_radial_outputs,
    save_random_walk_outputs,
    write_json,
)


def show_saved_figures(paths, *names):
    for name in names:
        path = paths[name]
        if Image is None:
            print(path)
        else:
            display(Image(filename=str(path)))

print(f"Notebook outputs: {OUT}")

## 1. Random walk to diffusion

Molecules move by many small thermal steps. The mean squared displacement links particle motion to the diffusion coefficient.

In [ ]:
# Parameters: random walk
particles = 1200
steps = 200
D = 80.0
dt = 0.01

rw = simulate_random_walk(particles=particles, steps=steps, D_um2_s=D, dt_s=dt, dimensions=2)
rw_paths = save_random_walk_outputs(rw, OUT / "random_walk")
write_json(rw.to_json_summary(), OUT / "random_walk" / "statistics.json")
show_saved_figures(rw_paths, "particle_cloud", "histogram", "msd")
rw.to_json_summary()["metadata"]

## 2. Fick's law and flux

A localized molecular release spreads down its concentration gradient. With no source or sink, no-flux walls approximately conserve total mass.

In [ ]:
# Parameters: Fickian diffusion
D = 100.0
total_time = 0.6

fick = simulate_cartesian_preset(preset="ficks_law_diffusion", D_um2_s=D, total_time_s=total_time)
fick_paths = save_cartesian_outputs(fick, OUT / "ficks_law_diffusion")
show_saved_figures(fick_paths, "time_evolution_panel", "statistics")
fick.state_label, fick.total_mass[0], fick.total_mass[-1]

## 3. Advection-diffusion in a microchannel

A microfluidic assay can carry target molecules downstream while diffusion spreads the plume across the channel.

In [ ]:
# Parameters: microchannel plume
D = 80.0
U = 200.0
total_time = 0.8

micro = simulate_cartesian_preset(
    preset="microchannel_advection_diffusion",
    D_um2_s=D,
    U_um_s=U,
    k_s=0.0,
    total_time_s=total_time,
)
micro_paths = save_cartesian_outputs(micro, OUT / "microchannel_advection_diffusion")
show_saved_figures(micro_paths, "time_evolution_panel", "velocity")
micro.dimensionless["Pe"]

## 4. Reaction-diffusion

Nutrients, oxygen, ligands, and drugs can be consumed or degraded while diffusing.

In [ ]:
# Parameters: reaction-diffusion sink
D = 80.0
k = 0.03
source_x = 24.0
sensor_x = 170.0

reaction = simulate_cartesian_preset(
    preset="reaction_diffusion_sink",
    D_um2_s=D,
    U_um_s=0.0,
    k_s=k,
    source_x_um=source_x,
    sensor_x_um=sensor_x,
    total_time_s=0.6,
)
reaction_paths = save_cartesian_outputs(reaction, OUT / "reaction_diffusion_sink")
show_saved_figures(reaction_paths, "time_evolution_panel", "statistics")
reaction.dimensionless["Da_diffusion"]

## 5. Cylindrical radial transport

Cylindrical coordinates are useful for transport around blood vessels, fibers, and long microchannel features. The reported flux is per unit length.

In [ ]:
# Parameters: cylindrical vessel transport
radius = 20.0
D = 120.0

cyl = simulate_radial_preset(
    preset="cylindrical_vessel_transport",
    geometry="cylindrical",
    D_um2_s=D,
    radius_um=radius,
    boundary_kind="noflux",
    total_time_s=0.3,
)
cyl_paths = save_radial_outputs(cyl, OUT / "cylindrical_vessel_transport")
show_saved_figures(cyl_paths, "time_evolution_panel", "statistics")
cyl.metadata["tau_diffusion_s"], cyl.boundary_flux[-1]

## 6. Spherical radial transport

Spherical coordinates describe cells, spheroids, aggregates, and drug-loaded beads. The spherical boundary flux includes the full surface area.

In [ ]:
# Parameters: spherical cell uptake
radius = 10.0
D = 100.0
outer_concentration = 1.0

sphere = simulate_radial_preset(
    preset="spherical_cell_uptake",
    geometry="spherical",
    D_um2_s=D,
    radius_um=radius,
    outer_concentration=outer_concentration,
    boundary_kind="absorbing",
    total_time_s=0.2,
)
sphere_paths = save_radial_outputs(sphere, OUT / "spherical_cell_uptake")
show_saved_figures(sphere_paths, "time_evolution_panel", "statistics")
sphere.state_label, sphere.boundary_flux[-1]

## 7. Microchannel biosensor

A sensor patch can absorb or bind target molecules. The sensor response curve is a simplified version of a biosensor breakthrough curve.

In [ ]:
# Parameters: biosensor design
D = 80.0
U = 200.0
k = 0.02
sensor_y = 18.0

biosensor = simulate_cartesian_preset(
    preset="microchannel_biosensor",
    D_um2_s=D,
    U_um_s=U,
    k_s=k,
    sensor_y_um=sensor_y,
    total_time_s=0.8,
)
biosensor_paths = save_cartesian_outputs(biosensor, OUT / "microchannel_biosensor")
show_saved_figures(biosensor_paths, "time_evolution_panel", "statistics")
biosensor.sensor_concentration[-1], biosensor.sensor_absorption_flux[-1]

## 8. Dimensionless interpretation

`Pe` compares advection with diffusion, `Da` compares reaction with transport, `Re` estimates microflow inertia, and the Bi-like number compares surface reaction with diffusion.

In [ ]:
# Parameters: dimensionless numbers
numbers = summarize_dimensionless_numbers(
    U_um_s=200.0,
    L_um=100.0,
    D_um2_s=80.0,
    k_s=0.02,
    surface_rate_um_s=1.0,
)

dimensionless_path = OUT / "dimensionless_numbers.png"
labels = list(numbers)
values = [float(item["value"]) for item in numbers.values()]

fig, ax = plt.subplots(figsize=(6.0, 3.4), constrained_layout=True)
ax.bar(labels, values, color=["#3b7ea1", "#d05a3f", "#5f8f4e", "#8b6bb3", "#c28f2c", "#4d6f8f"])
ax.set_yscale("log")
ax.set_title("Dimensionless number scale")
ax.set_ylabel("value (log scale)")
ax.tick_params(axis="x", labelrotation=35)
fig.savefig(dimensionless_path, dpi=160)
if Image is None:
    print(dimensionless_path)
else:
    display(Image(filename=str(dimensionless_path)))
plt.close(fig)
numbers

## 9. Student exercises

Change `D`, `U`, `k`, source position, sensor position, radius, geometry, and boundary condition. Explain what biological change each parameter represents and whether the response is diffusion-limited, advection-limited, or reaction-limited.

## 10. Limitations

The Python solver uses prescribed flow fields, 2D approximations, simple first-order reactions, and classroom-sized grids. Some examples do not reach a true steady state over the chosen simulation time; those outputs are labeled as final simulated state. Use the optional OpenFOAM backend when full CFD detail is needed.